In [1]:
from pathlib import Path
import pandas as pd

base = Path("../data")

cdc = pd.read_csv(
    base / "raw/cdc_homicides_2019_2024.csv"
)

ccw = pd.read_csv(
    base / "processed/rand_ccw_2019_2024.csv"
)

background = pd.read_csv(
    base / "processed/rand_background_2019_2024.csv"
)

print("CDC:", cdc.shape)
print("CCW:", ccw.shape)
print("Background:", background.shape)

CDC: (306, 8)
CCW: (300, 6)
Background: (300, 4)


In [2]:
cdc = cdc[cdc["name"] != "District of Columbia"].copy()

cdc = cdc.rename(columns={
    "name": "state",
    "period": "year",
    "count_sup": "homicide_count",
    "rate": "homicide_rate"
})

cdc["year"] = cdc["year"].astype(int)

print("CDC após padronização:", cdc.shape)

display(
    cdc[
        ["state", "year", "homicide_count", "homicide_rate"]
    ].head()
)

CDC após padronização: (300, 8)


,state,year,homicide_count,homicide_rate
0,Alabama,2019,587,12.0
1,Alabama,2020,654,13.0
2,Alabama,2021,748,14.8
3,Alabama,2022,706,13.9
4,Alabama,2023,717,14.0


In [3]:
chaves = ["state", "year"]

bases = {
    "CDC": cdc,
    "CCW": ccw,
    "Background": background
}

for nome, tabela in bases.items():
    print(f"\n{nome}")
    print("Registros:", len(tabela))
    print("Estados:", tabela["state"].nunique())
    print("Duplicações:", tabela.duplicated(chaves).sum())
    print("Chaves ausentes:", tabela[chaves].isna().sum().sum())

chaves_cdc = set(map(tuple, cdc[chaves].to_numpy()))
chaves_ccw = set(map(tuple, ccw[chaves].to_numpy()))
chaves_background = set(map(tuple, background[chaves].to_numpy()))

print("\nChaves CDC ausentes no CCW:", len(chaves_cdc - chaves_ccw))
print("Chaves CDC ausentes no Background:", len(chaves_cdc - chaves_background))
print("Chaves extras no CCW:", len(chaves_ccw - chaves_cdc))
print("Chaves extras no Background:", len(chaves_background - chaves_cdc))


CDC
Registros: 300
Estados: 50
Duplicações: 0
Chaves ausentes: 0

CCW
Registros: 300
Estados: 50
Duplicações: 0
Chaves ausentes: 0

Background
Registros: 300
Estados: 50
Duplicações: 0
Chaves ausentes: 0

Chaves CDC ausentes no CCW: 0
Chaves CDC ausentes no Background: 0
Chaves extras no CCW: 0
Chaves extras no Background: 0


In [4]:
df_integrado = (
    cdc.merge(
        ccw,
        on=["state", "year"],
        how="inner",
        validate="one_to_one"
    )
    .merge(
        background,
        on=["state", "year"],
        how="inner",
        validate="one_to_one"
    )
)

print("Dimensões:", df_integrado.shape)

display(df_integrado.head())

Dimensões: (300, 14)


,geoid,state,intent,year,homicide_count,homicide_rate,data_as_of,ttm_date_range,ccw_shall_issue,ccw_permitless,ccw_may_issue,ccw_prohibited,background_handgun,background_longgun
0,1,Alabama,All_Homicide,2019,587,12.0,2026-06-22T00:00:00.000,NaN,1,0,0,0,0,0
1,1,Alabama,All_Homicide,2020,654,13.0,2026-06-22T00:00:00.000,NaN,1,0,0,0,0,0
2,1,Alabama,All_Homicide,2021,748,14.8,2026-06-22T00:00:00.000,NaN,1,0,0,0,0,0
3,1,Alabama,All_Homicide,2022,706,13.9,2026-06-22T00:00:00.000,NaN,1,0,0,0,0,0
4,1,Alabama,All_Homicide,2023,717,14.0,2026-06-22T00:00:00.000,NaN,0,1,0,0,0,0


In [5]:
print("Registros:", len(df_integrado))

print("Estados:", df_integrado["state"].nunique())

print("Duplicações:",
      df_integrado.duplicated(["state", "year"]).sum())

print("\nValores ausentes:")
display(df_integrado.isna().sum())

Registros: 300
Estados: 50
Duplicações: 0

Valores ausentes:


geoid                   0
state                   0
intent                  0
year                    0
homicide_count          0
homicide_rate           0
data_as_of              0
ttm_date_range        300
ccw_shall_issue         0
ccw_permitless          0
ccw_may_issue           0
ccw_prohibited          0
background_handgun      0
background_longgun      0
dtype: int64

In [6]:
colunas = [
    "state",
    "year",
    "homicide_count",
    "homicide_rate",
    "ccw_shall_issue",
    "ccw_permitless",
    "ccw_may_issue",
    "ccw_prohibited",
    "background_handgun",
    "background_longgun"
]

df_modelo = df_integrado[colunas].copy()

df_modelo = df_modelo.sort_values(
    ["state", "year"]
).reset_index(drop=True)

print("Dimensões:", df_modelo.shape)
display(df_modelo.head(10))

Dimensões: (300, 10)


,state,year,homicide_count,homicide_rate,ccw_shall_issue,ccw_permitless,ccw_may_issue,ccw_prohibited,background_handgun,background_longgun
0,Alabama,2019,587,12.0,1,0,0,0,0,0
1,Alabama,2020,654,13.0,1,0,0,0,0,0
2,Alabama,2021,748,14.8,1,0,0,0,0,0
3,Alabama,2022,706,13.9,1,0,0,0,0,0
4,Alabama,2023,717,14.0,0,1,0,0,0,0
5,Alabama,2024,645,12.6,0,1,0,0,0,0
6,Alaska,2019,78,10.7,0,1,0,0,0,0
7,Alaska,2020,55,7.5,0,1,0,0,0,0
8,Alaska,2021,49,6.7,0,1,0,0,0,0
9,Alaska,2022,76,10.4,0,1,0,0,0,0


In [7]:
df_modelo["homicide_rate_next_year"] = (
    df_modelo
    .groupby("state")["homicide_rate"]
    .shift(-1)
)

display(
    df_modelo[
        ["state", "year", "homicide_rate",
         "homicide_rate_next_year"]
    ].head(12)
)

print(
    "Alvos ausentes:",
    df_modelo["homicide_rate_next_year"].isna().sum()
)

,state,year,homicide_rate,homicide_rate_next_year
0,Alabama,2019,12.0,13.0
1,Alabama,2020,13.0,14.8
2,Alabama,2021,14.8,13.9
3,Alabama,2022,13.9,14.0
4,Alabama,2023,14.0,12.6
5,Alabama,2024,12.6,NaN
6,Alaska,2019,10.7,7.5
7,Alaska,2020,7.5,6.7
8,Alaska,2021,6.7,10.4
9,Alaska,2022,10.4,8.3


Alvos ausentes: 50


In [8]:
df_modelo["next_year"] = (
    df_modelo.groupby("state")["year"].shift(-1)
)

intervalos_invalidos = df_modelo[
    df_modelo["next_year"].notna()
    & (df_modelo["next_year"] != df_modelo["year"] + 1)
]

print("Intervalos inválidos:", len(intervalos_invalidos))

display(intervalos_invalidos[
    ["state", "year", "next_year"]
].head(10))

Intervalos inválidos: 0


,state,year,next_year


In [9]:
# Manter apenas registros com variável-alvo conhecida
df_supervisionado = df_modelo.dropna(
    subset=["homicide_rate_next_year"]
).copy()

# Remover a coluna auxiliar de validação
df_supervisionado = df_supervisionado.drop(
    columns=["next_year"]
)

print("Dimensões:", df_supervisionado.shape)

display(df_supervisionado.head())

Dimensões: (250, 11)


,state,year,homicide_count,homicide_rate,ccw_shall_issue,ccw_permitless,ccw_may_issue,ccw_prohibited,background_handgun,background_longgun,homicide_rate_next_year
0,Alabama,2019,587,12.0,1,0,0,0,0,0,13.0
1,Alabama,2020,654,13.0,1,0,0,0,0,0,14.8
2,Alabama,2021,748,14.8,1,0,0,0,0,0,13.9
3,Alabama,2022,706,13.9,1,0,0,0,0,0,14.0
4,Alabama,2023,717,14.0,0,1,0,0,0,0,12.6


In [10]:
from pathlib import Path

destino = Path(
    "../data/processed/us_gun_policy_supervised_v1.csv"
)

df_supervisionado.to_csv(destino, index=False)

print("Salvo em:", destino.resolve())

Salvo em: /media/jandersongustavo/f8223ea0-bc74-41e8-8f66-650dbe31ee07/Arquivos_Janderson/Documentos/Projetos/us-gun-policy-ml/data/processed/us_gun_policy_supervised_v1.csv


In [12]:
from pathlib import Path
import pandas as pd

arquivo = Path("../data/raw/staadata/staadata.xlsx")

excel = pd.ExcelFile(arquivo)

print("Abas disponíveis:")
print(excel.sheet_names)

for aba in excel.sheet_names:
    amostra = pd.read_excel(
        arquivo,
        sheet_name=aba,
        nrows=8,
        header=None
    )

    print(f"\nAba: {aba}")
    display(amostra)

Abas disponíveis:
['staadata']

Aba: staadata


,0,1,2,3,4,5,6,7,8,9
0,States and selected areas: Employment status ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1976 to 2025(1) annual averages,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FIPS Code,State and area,Year,Civilian non-institutional population,Civilian labor force,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,Total,Percent of population,Employment,NaN,Unemployment,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,Total,Percent of population,Total,Rate
6,NaN,NaN,NaN,NaN,NaN,NaN,Labor force,NaN,level,rate
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
bls = pd.read_excel(
    arquivo,
    sheet_name=excel.sheet_names[0],
    header=None,
    skiprows=8,
    usecols=[0, 1, 2, 9],
    names=["fips", "state", "year", "unemployment_rate"]
)

print("Dimensões:", bls.shape)

display(bls.head(10))

print("\nTipos de dados:")
print(bls.dtypes)

Dimensões: (2652, 4)


,fips,state,year,unemployment_rate
0,01,Alabama,1976,6.7
1,02,Alaska,1976,7.7
2,04,Arizona,1976,9.8
3,05,Arkansas,1976,6.9
4,06,California,1976,9.2
5,037,Los Angeles County,1976,8.9
6,08,Colorado,1976,6.0
7,09,Connecticut,1976,9.2
8,10,Delaware,1976,8.7
9,11,District of Columbia,1976,8.7



Tipos de dados:
fips                     str
state                    str
year                     str
unemployment_rate    float64
dtype: object


In [17]:
# Utilizar os estados da base CDC como referência
cdc = pd.read_csv(
    "../data/raw/cdc_homicides_2019_2024.csv"
)

estados_validos = set(cdc["name"]) - {"District of Columbia"}

bls["year"] = pd.to_numeric(
    bls["year"],
    errors="coerce"
)

bls_filtrado = bls[
    bls["state"].isin(estados_validos)
    & bls["year"].between(2019, 2024)
].copy()

print("Dimensões:", bls_filtrado.shape)
print("Estados:", bls_filtrado["state"].nunique())

display(bls_filtrado.groupby("year")["state"].nunique())

Dimensões: (300, 4)
Estados: 50


year
2019.0    50
2020.0    50
2021.0    50
2022.0    50
2023.0    50
2024.0    50
Name: state, dtype: int64

In [18]:
estados_validos = set(cdc["name"]) - {"District of Columbia"}

bls_filtrado = bls[
    bls["state"].isin(estados_validos)
    & bls["year"].between(2019, 2024)
].copy()

print("Dimensões:", bls_filtrado.shape)
print("Estados:", bls_filtrado["state"].nunique())

display(bls_filtrado.groupby("year")["state"].nunique())

Dimensões: (300, 4)
Estados: 50


year
2019.0    50
2020.0    50
2021.0    50
2022.0    50
2023.0    50
2024.0    50
Name: state, dtype: int64

In [19]:
# Validação final da base BLS
print("Duplicações:",
      bls_filtrado.duplicated(["state", "year"]).sum())

print("Taxas ausentes:",
      bls_filtrado["unemployment_rate"].isna().sum())

print("Taxas fora do intervalo:",
      (~bls_filtrado["unemployment_rate"].between(0, 100)).sum())

print("\nEstatísticas descritivas:")
display(bls_filtrado["unemployment_rate"].describe())

Duplicações: 0
Taxas ausentes: 0
Taxas fora do intervalo: 0

Estatísticas descritivas:


count    300.000000
mean       4.334000
std        1.810166
min        1.800000
25%        3.100000
50%        3.900000
75%        5.025000
max       13.500000
Name: unemployment_rate, dtype: float64

In [20]:
from pathlib import Path

bls_final = bls_filtrado[
    ["state", "year", "unemployment_rate"]
].copy()

bls_final["year"] = bls_final["year"].astype(int)

destino = Path("../data/processed/bls_unemployment_2019_2024.csv")

bls_final.to_csv(destino, index=False)

print("Dimensões:", bls_final.shape)
print("Arquivo salvo:", destino.resolve())

display(bls_final.head())

Dimensões: (300, 3)
Arquivo salvo: /media/jandersongustavo/f8223ea0-bc74-41e8-8f66-650dbe31ee07/Arquivos_Janderson/Documentos/Projetos/us-gun-policy-ml/data/processed/bls_unemployment_2019_2024.csv


,state,year,unemployment_rate
2279,Alabama,2019,3.2
2280,Alaska,2019,5.6
2281,Arizona,2019,4.8
2282,Arkansas,2019,3.5
2283,California,2019,4.1


In [21]:
bls = pd.read_csv(
    "../data/processed/bls_unemployment_2019_2024.csv"
)

df_v2 = df_supervisionado.merge(
    bls,
    on=["state", "year"],
    how="left",
    validate="one_to_one"
)

print("Dimensões:", df_v2.shape)
print("Desemprego ausente:", df_v2["unemployment_rate"].isna().sum())

display(
    df_v2[
        ["state", "year", "homicide_rate",
         "unemployment_rate", "homicide_rate_next_year"]
    ].head(10)
)

Dimensões: (250, 12)
Desemprego ausente: 0


,state,year,homicide_rate,unemployment_rate,homicide_rate_next_year
0,Alabama,2019,12.0,3.2,13.0
1,Alabama,2020,13.0,6.4,14.8
2,Alabama,2021,14.8,3.4,13.9
3,Alabama,2022,13.9,2.5,14.0
4,Alabama,2023,14.0,2.5,12.6
5,Alaska,2019,10.7,5.6,7.5
6,Alaska,2020,7.5,8.3,6.7
7,Alaska,2021,6.7,6.4,10.4
8,Alaska,2022,10.4,4.4,8.3
9,Alaska,2023,8.3,4.3,7.6


In [22]:
from pathlib import Path

destino = Path("../data/processed/us_gun_policy_supervised_v2.csv")

df_v2.to_csv(destino, index=False)

print("Arquivo salvo:", destino.resolve())

Arquivo salvo: /media/jandersongustavo/f8223ea0-bc74-41e8-8f66-650dbe31ee07/Arquivos_Janderson/Documentos/Projetos/us-gun-policy-ml/data/processed/us_gun_policy_supervised_v2.csv


In [23]:
bea_income = pd.read_csv(
    "../data/processed/bea_income_2019_2024.csv"
)

df_v3 = df_v2.merge(
    bea_income,
    on=["state", "year"],
    how="left",
    validate="one_to_one"
)

print("Dimensões:", df_v3.shape)
print("Rendas ausentes:", df_v3["income_per_capita"].isna().sum())

display(
    df_v3[
        [
            "state",
            "year",
            "homicide_rate",
            "unemployment_rate",
            "income_per_capita",
            "homicide_rate_next_year"
        ]
    ].head(10)
)

Dimensões: (250, 13)
Rendas ausentes: 0


,state,year,homicide_rate,unemployment_rate,income_per_capita,homicide_rate_next_year
0,Alabama,2019,12.0,3.2,42996.0,13.0
1,Alabama,2020,13.0,6.4,45874.0,14.8
2,Alabama,2021,14.8,3.4,50651.0,13.9
3,Alabama,2022,13.9,2.5,51706.0,14.0
4,Alabama,2023,14.0,2.5,54760.0,12.6
5,Alaska,2019,10.7,5.6,60483.0,7.5
6,Alaska,2020,7.5,8.3,61892.0,6.7
7,Alaska,2021,6.7,6.4,65773.0,10.4
8,Alaska,2022,10.4,4.4,69151.0,8.3
9,Alaska,2023,8.3,4.3,72563.0,7.6


In [24]:
print("Dimensões:", df_v3.shape)

print("Duplicações:",
      df_v3.duplicated(["state", "year"]).sum())

print("Valores ausentes por coluna:")
display(df_v3.isna().sum())

print("Estados:", df_v3["state"].nunique())
print("Anos:", sorted(df_v3["year"].unique()))

Dimensões: (250, 13)
Duplicações: 0
Valores ausentes por coluna:


state                      0
year                       0
homicide_count             0
homicide_rate              0
ccw_shall_issue            0
ccw_permitless             0
ccw_may_issue              0
ccw_prohibited             0
background_handgun         0
background_longgun         0
homicide_rate_next_year    0
unemployment_rate          0
income_per_capita          0
dtype: int64

Estados: 50
Anos: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


In [25]:
from pathlib import Path

destino = Path("../data/processed/us_gun_policy_supervised_v3.csv")

df_v3.to_csv(destino, index=False)

print("Dataset V3 salvo:", destino.resolve())
print("Dimensões:", df_v3.shape)

Dataset V3 salvo: /media/jandersongustavo/f8223ea0-bc74-41e8-8f66-650dbe31ee07/Arquivos_Janderson/Documentos/Projetos/us-gun-policy-ml/data/processed/us_gun_policy_supervised_v3.csv
Dimensões: (250, 13)
